In [1]:
from __future__ import division
%matplotlib inline
import pandas as pd
import numpy as np
import sys
from collections import defaultdict
import matplotlib.pyplot as plt
from copy import deepcopy
import csv
# add the deep_disfluency module to the path
sys.path.append("../../../..")

In [2]:
from deep_disfluency.evaluation.disf_evaluation import incremental_output_disfluency_eval_from_file
from deep_disfluency.evaluation.disf_evaluation import final_output_disfluency_eval_from_file

from deep_disfluency.evaluation.eval_utils import rename_all_repairs_in_line_with_index

from deep_disfluency.evaluation.results_utils import convert_to_latex

In [3]:
# first correct the hypothesis file to make sure it has consistent tabs throughout
hyp_file = open("result-1-corrected.txt")
final_hyp_file = open("result-1-corrected-tabs.txt", "w")
for line in hyp_file:
    data = line.strip("\n").split("\t")
    if len(data) < 5:
        data = [""] + data
    final_hyp_file.write("\t".join(data) + "\n")
hyp_file.close()
final_hyp_file.close()

In [4]:
# first convert files to 'timings' file format for evaluation purposes'

In [5]:
def add_word_continuation_tags(tags, ground_truth):
    """Returns list with continuation tags for each word:
    <cc/> continues current dialogue act and the next word will also continue
    <ct/> continues current dialogue act and is the last word of it
    <tc/> starts this dialogue act tag and the next word continues it
    <tt/> starts and ends dialogue act (single word dialogue act)
    """
    tags = list(tags)
    for i in range(0, len(tags)):
        if not ground_truth:
            pred_tag = tags[i].split("@")[1]
            tags[i] = tags[i].split("@")[0]
        else:
            tags[i] = tags[i].split("@")[0]
        if i == 0:
            tags[i] = tags[i] + "<t"
        else:
            tags[i] = tags[i] + "<c"
        if i == len(tags)-1:
            tags[i] = tags[i] + "t/>"
        else:
            tags[i] = tags[i] + "c/>"
        if not ground_truth:
            tags[i] = tags[i] + "@" + pred_tag
    return tags

def get_tag_data_from_corpus_file(f, ground_truth):
    """Loads from file into four lists of lists of strings of equal length:
    one for utterance iDs (IDs))
    one for words (seq), 
    one for pos (pos_seq) 
    one for tags (targets)."""

    f = open(f)
    print "loading data", f.name
    count_seq = 0
    IDs = []
    seq = []
    pos_seq = []
    targets = []
    mappings = []

    reader = csv.reader(f, delimiter='\t')
    counter = 0
    utt_reference = ""
    currentWords = []
    currentPOS = []
    currentTags = []
    currentMappings = []

    # corpus = "" # can write to file
    for ref, map, word, postag, disftag in reader:  # mixture of POS and Words
        counter += 1
        if not ref == "":
            if count_seq > 0:  # do not reset the first time
                # convert to the inc tags
                # corpus+=utt_reference #write data to a file for checking
                # convert to vectors
                seq.append(tuple(currentWords))
                pos_seq.append(tuple(currentPOS))
                targets.append(tuple(add_word_continuation_tags(currentTags,
                                                               ground_truth=ground_truth)))
                IDs.append(utt_reference)
                mappings.append(tuple(currentMappings))
                # reset the words
                currentWords = []
                currentPOS = []
                currentTags = []
                currentMappings = []
            # set the utterance reference
            count_seq += 1
            utt_reference = ref
        currentWords.append(word)
        currentPOS.append(postag)
        currentTags.append(disftag)
        currentMappings.append(map)
    # flush
    if not currentWords == []:
        seq.append(tuple(currentWords))
        pos_seq.append(tuple(currentPOS))
        targets.append(tuple(add_word_continuation_tags(currentTags,
                                                       ground_truth=ground_truth)))
        IDs.append(utt_reference)
        mappings.append(tuple(currentMappings))

    assert len(seq) == len(targets) == len(pos_seq)
    print "loaded " + str(len(seq)) + " sequences"
    f.close()
    return (IDs, mappings, seq, pos_seq, targets)


def sort_into_dialogue_speakers(IDs, mappings, utts, pos_tags=None,
                                labels=None):
    """For each utterance, given its ID get its conversation number and
    dialogue participant in the format needed for word alignment files.

    Returns a list of tuples:
    (speaker, mappings, utts, pos, labels)

    """
    dialogue_speaker_dict = dict()  # keys are speaker IDs of filename:speaker
    # vals are tuples of (mappings, utts, pos_tags, labels)
    current_speaker = ""

    for ID, mapping, utt, pos, label in zip(IDs,
                                            mappings,
                                            utts,
                                            pos_tags,
                                            labels):
        split = ID.split(":")
        dialogue = split[0]
        speaker = split[1]
        # uttID = split[2]
        current_speaker = "-".join([dialogue, speaker])
        if current_speaker not in dialogue_speaker_dict.keys():
            dialogue_speaker_dict[current_speaker] = [[], [], [], []]

        dialogue_speaker_dict[current_speaker][0].extend(list(mapping))
        dialogue_speaker_dict[current_speaker][1].extend(list(utt))
        dialogue_speaker_dict[current_speaker][2].extend(list(pos))
        dialogue_speaker_dict[current_speaker][3].extend(list(label))
    # turn into 5-tuples
    dialogue_speakers = [(key,
                          dialogue_speaker_dict[key][0],
                          dialogue_speaker_dict[key][1],
                          dialogue_speaker_dict[key][2],
                          dialogue_speaker_dict[key][3])
                         for key in sorted(dialogue_speaker_dict.keys())]
    return dialogue_speakers


def write_corpus_file_add_fake_timings_and_utt_tags(f, target_path,
                                                    ground_truth=True,
                                                    verbose=False):
    target_file = open(target_path, "w")
    IDs, mappings, utts, pos_tags, labels = get_tag_data_from_corpus_file(f,
                                                            ground_truth=ground_truth)
    dialogue_speakers = sort_into_dialogue_speakers(IDs,
                                                    mappings,
                                                    utts,
                                                    pos_tags,
                                                    labels)
    for speaker_name, mapping, utt, pos, label in dialogue_speakers:
        if verbose:
            print "*" * 30
            print speaker_name
            print mapping
            print utt
            print pos
            print label
            y = raw_input()
            if y == "y":
                quit()
        target_file.write("Speaker: " + speaker_name + "\n")
        starts = range(0, len(label))
        ends = range(1, len(label)+1)
        for m, s, e, w, p, l in zip(mapping, starts, ends, utt, pos, label):
            if ground_truth:
                l = "\t".join([m, str(float(s)), str(float(e)), w, p, l])
            else:
                # no pos tags
                l = "\t".join([str(float(s)), str(float(e)), w, l])
            
            target_file.write(l + "\n")
        target_file.write("\n")
    target_file.close()


def convert_file_to_fake_timings_file(f, f_new_name, ground_truth=True):
    #f = "../../../stir/python/data/bnc_spoken/BNC-CH_partial_data.csv"
    #write_corpus_file_add_fake_timings_and_utt_tags(
    #    f, f.replace("_data", "_data_timings"))
    write_corpus_file_add_fake_timings_and_utt_tags(
        f, f_new_name, ground_truth=ground_truth)

In [7]:
IDs, mappings, utts, pos_tags, labels = get_tag_data_from_corpus_file("duel_corpus_ch.txt",
                                                            ground_truth=True)

loading data duel_corpus_ch.txt
loaded 13014 sequences


In [8]:
IDs2, mappings2, utts2, pos_tags2, labels2 = get_tag_data_from_corpus_file("result-1-corrected-tabs.txt",
                                                            ground_truth=True)

loading data result-1-corrected-tabs.txt
loaded 13012 sequences


In [9]:
# Make sure all the utterances are in there in both! Should be an empty set
set(IDs) - set(IDs2)

{'ch-r3-dream_apartment:B:224:X', 'ch-r3-dream_apartment:B:225:X'}

In [10]:
# make sure all the mappings/words are there - should have no mismatches!
mismatched = []
for _id, ground_truth_mapping, hyp_mapping in zip(IDs, mappings, mappings2):
    #print _id
    if not ground_truth_mapping == hyp_mapping:
        mismatched.append(_id)
print len(mismatched), "mismatched" 
# print mismatched  # find out which sequences are wrong

10403 mismatched


In [13]:
# add fake timings to result-1 file to make a fake ground truth file for illustration
# in the final experiment, the evaluation should only use the duel_corpus_ch_timings.txt file
convert_file_to_fake_timings_file("result-1-corrected-tabs.txt",
                                  "duel_corpus_ch_timings_fake.txt",
                                  ground_truth=True)

loading data result-1-corrected-tabs.txt
loaded 13012 sequences


In [14]:
# add fake timings to hypothesis file
convert_file_to_fake_timings_file("result-1-corrected-tabs.txt",
                                  "result-1-corrected-tabs_timings.txt",
                                  ground_truth=False)

loading data result-1-corrected-tabs.txt
loaded 13012 sequences


# Final output evaluation

In [16]:
# reload the old methods
from deep_disfluency.evaluation.eval_utils import get_tag_data_from_corpus_file
from deep_disfluency.evaluation.eval_utils import sort_into_dialogue_speakers

# ground truth files
# for now using fake file should really only use the duel_corpus_ch_timing.txt file!
ground_truth_files = [
                    "duel_corpus_ch_timings_fake.txt"
                     #"duel_corpus_ch_timings.txt"
                   ]

# names of the files with tags from systems
systems_hyp_files = ["result-1-corrected-tabs_timings.txt"]

In [17]:
print ground_truth_files
print systems_hyp_files

['duel_corpus_ch_timings_fake.txt']
['result-1-corrected-tabs_timings.txt']


In [18]:
all_results = {}
all_error_dicts = {}
VERBOSE = True
for hyp_file in systems_hyp_files:
    system = hyp_file.replace(".txt", "")
    #if 'complex' in system: break
    for division, disf_file in zip(["test"], ground_truth_files):

        print "*" * 30, division, "*" * 30
        IDs, timings, words, pos_tags, labels = get_tag_data_from_corpus_file(disf_file)
        gold_data = {} #map from the file name to the data
        for dialogue,a,b,c,d in zip(IDs, timings, words, pos_tags, labels):
            print dialogue
            d = rename_all_repairs_in_line_with_index(list(d))
            gold_data[dialogue] = (a,b,c,d)

        #the below does just the final output evaluation, assuming a final output file, faster
        word = True  # world-level analyses
        error = True # get an error analysis
        results,speaker_rate_dict,error_analysis = final_output_disfluency_eval_from_file(
                                                        hyp_file,
                                                        gold_data,
                                                        utt_eval=False,
                                                        error_analysis=error,
                                                        word=word,
                                                        interval=False,
                                                        outputfilename=None
                                                    )
        #the below does incremental and final output in one, also outputting the final outputs
        #derivable from the incremental output, takes quite a while
        if VERBOSE:
            for k,v in results.items():
                print k,v
        all_results[division + "_" + system] = deepcopy(results)
        #if "heldout" in division:
            # only do the error analyses on the heldout data
        if True:
            # collate for error analysis
            all_error_dicts[division + "_" + system] = deepcopy(error_analysis)


****************************** test ******************************
loading data duel_corpus_ch_timings_fake.txt
loaded 58 sequences
ch-r1-border_control-A
ch-r1-border_control-B
ch-r1-dream_apartment-A
ch-r1-dream_apartment-B
ch-r1-film_script-A
ch-r1-film_script-B
ch-r10-border_control-A
ch-r10-border_control-B
ch-r10-dream_apartment-A
ch-r10-dream_apartment-B
ch-r10-film_script-A
ch-r10-film_script-B
ch-r2-border_control-A
ch-r2-border_control-B
ch-r2-dream_apartment-A
ch-r2-dream_apartment-B
ch-r2-film_script-A
ch-r2-film_script-B
ch-r3-border_control-A
ch-r3-border_control-B
ch-r3-dream_apartment-A
ch-r3-dream_apartment-B
ch-r3-film_script-A
ch-r3-film_script-B
ch-r4-border_control-A
ch-r4-border_control-B
ch-r4-dream_apartment-A
ch-r4-dream_apartment-B
ch-r4-film_script-A
ch-r4-film_script-B
ch-r5-border_control-A
ch-r5-border_control-B
ch-r5-dream_apartment-A
ch-r5-dream_apartment-B
ch-r6-border_control-A
ch-r6-border_control-B
ch-r6-dream_apartment-A
ch-r6-dream_apartment-B
ch-r

In [19]:
print all_results.keys()

['test_result-1-corrected-tabs_timings']


In [20]:
# you can add different results here
display_results = dict()
display_results['LSTM (window length=2) (+ POS)'] = all_results['test_result-1-corrected-tabs_timings']
#display_results['BI-LSTM (window length=2) (+ POS)'] = all_results['test-result-2-corrected-tabs_timings']
final = convert_to_latex(display_results, eval_level=['word'], inc=False, utt_seg=False, only_include=
                        ['f1_<rm_word', 'f1_<rps_word', 'f1_<e_word'])
#final = final.drop(final.columns[[-2]], axis=1)
final

,System (eval. method),$F_{rm}$ (per word),$F_{rps}$ (per word),$F_{e}$ (per word)
0,LSTM (window length=2) (+ POS) (transcript),0.004,0.005,0.767


# Error Analysis

In [21]:
#Error analyses on exact match ('rms') and getting the right repair start ('rps')
target_tags = [
    #'<rms',
    '<rps',
    '<e'
    ]

for div,all_error in all_error_dicts.items():
    # print div, type(all_error)
   
    if type(all_error) == bool: continue
    #if "test" in div: continue
    #if not 'TTO only' in div or "asr" in div: continue
    for tag, errors in all_error.items():
        if tag not in target_tags:
            continue
        print "*" * 30, div, tag, "*" * 30
        # print errors
        # continue
        #if not 'TTO only' in div or "asr" in div: continue
        error = {"TP" : {}, "FP" : {}, "FN": {} }
        for k,v in errors.items():
            #if k == "FP":
            #    continue
            print k, len(v)
            typedict = defaultdict(int)
            lendict = defaultdict(int)
            for repair in v:

                #print repair.gold_context
                onset = ""
                if tag == "<rps" or tag == "<rms":
                    
                    
                    for i in range(0,len(repair.gold_context)):
                        if repair.gold_context[i] == "+|+":
                            onset = repair.gold_context[i+1]
                            break

                    word = onset.split("|")[0]
                    #if k == "FP":
                    #    onset = gold_onset
                    if "<e" in onset and not tag == "<e":
                        typedict["<e"]+=1
                    else:
                        if word in ["and","or","but","so","because","that","although"]:
                            typedict["CC"]+=1
                        elif word in ["i","we","they","im","ive","he","she","id"]:
                            typedict["subj"]+=1
                        elif word in ["you","the"] or "$" in word:
                            typedict["proper_other"]+=1
                        elif word in ["yeah","no","okay","yes","right","uh-huh"]:
                            typedict["ack"]+=1
                        elif word in ["it","its"]:
                            typedict["it"]+=1
                        else:
                            typedict[word]+=1
                
                if tag == "<rps" or tag == "<rms": # and not k == 'FP':
                    if k == "TP" and len(repair.reparandumWords) > 8:
                        # should not be getting any over 8 words
                        print "** overlength repair!"
                        print repair
                    lendict[len(repair.reparandumWords) + len(repair.interregnumWords)]+=1
                    repair_type = None
                    if repair.type:
                        repair_type = repair.type 
                        typedict[repair_type]+=1

            error[k]['len'] = deepcopy(lendict)
            error[k]['type'] = deepcopy(typedict)

        if tag == "<e": 
            continue
        for mode in ['type', 'len']:
            #q1. THE RECALL RATES FOR VARIOUS GOLD REPAIR TYPES
            print mode, "*" * 30
            tps = error['TP'][mode]
            fns = error['FN'][mode]
            fps = error['FP'][mode]

            total_tps = 0
            total_fns = 0
            total_fps = 0
            top_n = 50
            all_items = list(set(tps.keys() + fns.keys()))
            # print all_items
            for k in sorted(all_items,  reverse=False):
                #print k, "*" * 30
                if mode == 'type' and k not in ["rep", "del", "sub"]:
                    continue
                recall_total = tps[k] + fns[k]
                recall = 0 if tps[k] == 0 else tps[k]/recall_total
                precision_total = tps[k] + fps[k]
                precision = 0 if tps[k] == 0 else tps[k]/precision_total
                fscore = 0 if precision == 0 or recall == 0 else (2 * (precision * recall))/(precision + recall)
                # print k, ':', tps[k], "out of", recall_total
                #print k, ':', tps[k], "out of", precision_total
                total_tps += tps[k]
                total_fns += fns[k]
                total_fps += fps[k]
                print " & ".join([str(k), "({0}/{1})".format(tps[k],recall_total), 
                                  '{0:.3f}'.format(fscore)]) + "\\\\"
                top_n-=1
                if top_n <= 0:
                    break
            print total_tps/(total_fns + total_tps)

            if False:
                #q2. ERROR TYPE SUMMARY
                print "*" * 30
                total = sum(fns.values()+tps.values())

                errormass = 0
                errortotal = 0
                top_n = 20
                for k,v in sorted(tps.items(),key= lambda x: x[1],reverse=True):
                    print k,"&",v,"&",'{0:.2f}'.format(v/total)
                    errormass +=(v/total * 100)
                    errortotal+=v
                    top_n-=1
                    if top_n <= 0: break
                print "total &",errortotal,"&",'{0:.2f}'.format(errormass)

****************************** test_result-1-corrected-tabs_timings <rps ******************************
FP 31
TP 6
FN 2488
type ******************************
del & (2/225) & 0.018\\
rep & (0/744) & 0.000\\
sub & (3/1273) & 0.005\\
0.00223015165031
len ******************************
0 & (1/254) & 0.008\\
1 & (5/1151) & 0.008\\
2 & (0/570) & 0.000\\
3 & (0/252) & 0.000\\
4 & (0/141) & 0.000\\
5 & (0/69) & 0.000\\
6 & (0/31) & 0.000\\
7 & (0/10) & 0.000\\
8 & (0/16) & 0.000\\
0.00240577385726
****************************** test_result-1-corrected-tabs_timings <e ******************************
FP 1446
TP 4121
FN 1053
